# 🏛️ Step 1: 회의록 원본 JSON에서 본문 HTML 샘플 추출
본 노트북은 원본 지방의회 회의록 JSON 파일의 `MINTS_HTML` 필드로부터 대표적인 본문 발언 영역을 선별·추출하여 LLM 파싱 규칙 학습용 샘플 텍스트(`sample_*.txt`)를 생성합니다.

### 📌 주요 처리 과정
1. **HTML 정제 (`prepare_html`)**: 스크립트, 스타일 태그, 상하단 네비게이션/푸터 및 주석 제거
2. **발언 영역 탐색 (`find_transcript_region`)**: 발언자 기호(○, ◯ 등) 및 CSS 힌트를 바탕으로 최적의 본문 컨테이너 스코어링
3. **안전한 본문 축약 (`shorten_long_text`)**: 태그 계층 구조는 보존하면서 장문 텍스트 노드를 요약(300자)하여 LLM 컨텍스트 한도 내 유지
4. **샘플 저장 (`extract_samples_from_folder`)**: 각 의회별 대표 3건의 회의록 본문을 샘플 파일로 저장

In [ ]:
import glob
import html
import json
import os
import random
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup, Comment, Tag

# ---------------------------------------------------------------------------
# 1. 환경 설정 및 경로 지정
# ---------------------------------------------------------------------------
TARGET_YEAR = "2024"

# 원본 JSON 경로 후보 자동 탐색 (origin/ 우선)
candidate_dirs = [
    Path(f"../../origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"../origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"./origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"../../data/org/{TARGET_YEAR}_광역 회의록"),
    Path(f"./data/org/{TARGET_YEAR}_광역 회의록"),
]
INPUT_DIR = str(next((p for p in candidate_dirs if p.exists()), Path(f"./origin/{TARGET_YEAR}_광역 회의록")))
OUTPUT_DIR = f"./data/sample/{TARGET_YEAR}_광역 회의록"
NUM_RECORDS = 3  # 각 의회별 추출할 샘플 레코드 개수

print(f"입력 경로: {os.path.abspath(INPUT_DIR)}")
print(f"출력 경로: {os.path.abspath(OUTPUT_DIR)}")

## 2. 정규식 힌트 패턴 및 HTML 전처리 함수
HTML 내의 네비게이션, 푸터, 스크립트 등의 노이즈 요소를 제거하고 본문 후보 태그를 식별하기 위한 정규식을 정의합니다.

In [ ]:
# 본문 식별용 CSS 클래스/ID 힌트
TRANSCRIPT_HINT = re.compile(
    r"(?:sMan|speaker|speech|utterance|statement|minutes|record[-_]?content|record[-_]?body|transcript|view[-_]?content|con[-_]?view|an\d+)",
    re.IGNORECASE,
)

# 발언자 기호 및 직함 패턴
SPEECH_HINT = re.compile(
    r"(?:○\s*[^\n]{2,30}|◯\s*[^\n]{2,30}|(?:위원장|의장|의원|국장|과장|교육감|부위원장)\s*[가-힣]{2,5}\s*(?:의원|위원|:|\s))"
)

# 제거 대상 노이즈 클래스/ID 패턴
NOISE = re.compile(
    r"(?:sidebar|navbar|navigation|breadcrumb|footer|header|toolbar|menu|pagination|related|sns)",
    re.IGNORECASE,
)

def prepare_html(raw_html: str) -> BeautifulSoup:
    """HTML 엔티티 변환 및 노이즈 요소(스크립트, 스타일, 네비게이션 등) 제거"""
    if "&lt;" in raw_html or "&gt;" in raw_html:
        raw_html = html.unescape(raw_html)
    soup = BeautifulSoup(raw_html, "html.parser")
    
    # 주석 노드 제거
    for node in soup.find_all(string=lambda s: isinstance(s, Comment)):
        node.extract()
        
    # 불필요한 레이아웃/스크립트 태그 일괄 제거
    for node in soup(["script", "style", "noscript", "meta", "link", "header", "nav", "footer"]):
        node.decompose()

    # 노이즈 aside/form 제거
    for tag in list(soup.find_all(["aside", "form"])):
        if tag.name is None or not isinstance(tag.attrs, dict):
            continue
        attrs = " ".join(tag.get("class") or []) + " " + (tag.get("id") or "")
        if NOISE.search(attrs) or (tag.name == "aside" and not TRANSCRIPT_HINT.search(attrs)):
            tag.decompose()

    # 노이즈 태그 제거
    for tag in list(soup.find_all(True)):
        if tag.name is None or not isinstance(tag.attrs, dict):
            continue
        classes = tag.get("class") or []
        element_id = tag.get("id") or ""
        attrs = " ".join(classes) + " " + element_id
        if NOISE.search(attrs) and not TRANSCRIPT_HINT.search(attrs):
            tag.decompose()

    return soup

## 3. 본문 영역 탐색 및 긴 텍스트 축약 함수
발언 마커 빈도와 텍스트 밀도를 종합 채점하여 발언 본문이 밀집된 최적의 컨테이너를 탐색합니다.

In [ ]:
def find_transcript_region(soup: BeautifulSoup) -> Tag | None:
    """의미론적/구조적 증거를 기반으로 발언 본문 영역을 탐색합니다."""
    candidates = []
    for tag in soup.find_all(["main", "article", "section", "div", "table"]):
        txt = tag.get_text(" ", strip=True)
        if len(txt) < 120:
            continue
        attrs = " ".join(tag.get("class") or []) + " " + (tag.get("id") or "")
        marker_count = len(SPEECH_HINT.findall(txt))
        structure = bool(TRANSCRIPT_HINT.search(attrs))
        descendant_hint = len(tag.find_all(attrs={"class": TRANSCRIPT_HINT})) + len(
            tag.find_all(attrs={"id": TRANSCRIPT_HINT})
        )
        if not (marker_count >= 2 or descendant_hint >= 2 or (structure and marker_count >= 1)):
            continue
        # 발언 마커 수와 구조 힌트를 바탕으로 스코어링
        score = min(marker_count, 30) * 3 + min(descendant_hint, 30) * 2 + (8 if structure else 0)
        score -= min(len(txt) // 12000, 10)
        candidates.append((score, -len(str(tag)), tag))

    if not candidates:
        return None
    return max(candidates, key=lambda item: (item[0], item[1]))[2]

def shorten_long_text(region: Tag, limit: int = 300) -> str:
    """HTML 구조를 유지하면서 긴 텍스트 노드의 문자 수만 제한합니다."""
    copy_soup = BeautifulSoup(str(region), "html.parser")
    for node in list(copy_soup.find_all(string=True)):
        value = str(node)
        if len(value) > limit:
            node.replace_with(value[:limit] + "…[발언 내용 일부 생략]")
    return str(copy_soup)

## 4. 샘플 추출 실행
지정된 입력 디렉토리에서 JSON 파일들을 읽어 의회별 대표 본문 HTML 샘플을 추출합니다.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.json")))
if not files:
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "*.json"), recursive=True))

print(f"발견된 원본 회의록 JSON 파일 수: {len(files)}개")

success_count = 0
for path in files:
    base = os.path.splitext(os.path.basename(path))[0]
    # 빈 파일(예: 0바이트/2바이트) 건너뛰기
    if os.path.getsize(path) < 100:
        continue
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        records = data if isinstance(data, list) else [data]
        middle = records[5:-5] if len(records) > 10 else records
        indices = list(range(len(middle)))
        random.shuffle(indices)

        samples = []
        for idx in indices:
            record = middle[idx]
            if not isinstance(record, dict) or not isinstance(record.get("MINTS_HTML"), str):
                continue
            soup = prepare_html(record["MINTS_HTML"])
            region = find_transcript_region(soup)
            if region is None:
                continue
            samples.append(shorten_long_text(region))
            if len(samples) >= NUM_RECORDS:
                break

        if samples:
            out_file = os.path.join(OUTPUT_DIR, f"sample_{base}.txt")
            with open(out_file, "w", encoding="utf-8") as f:
                for i, sample in enumerate(samples, start=1):
                    f.write(f"--- [Random Sample Record {i} from {os.path.basename(path)}] ---\n")
                    f.write(sample + "\n\n")
            print(f"✓ [{base}] {len(samples)}건 샘플 저장 완료 -> {out_file}")
            success_count += 1
        else:
            print(f"⚠️ [{base}] 본문 후보를 찾지 못함")
    except Exception as exc:
        print(f"❌ [{base}] 오류 발생: {exc}")

print(f"\n총 {success_count}/{len(files)}개 의회 샘플 추출 완료!")

## 5. 생성된 샘플 파일 내용 확인
정상적으로 생성된 샘플 파일 중 하나를 열어 앞부분 구조를 확인합니다.

In [ ]:
sample_files = glob.glob(os.path.join(OUTPUT_DIR, "sample_*.txt"))
if sample_files:
    sample_preview = Path(sample_files[0]).read_text(encoding="utf-8")
    print(f"미리보기 대상: {sample_files[0]}")
    print("-" * 60)
    print(sample_preview[:1000] + "\n...")
else:
    print("생성된 샘플 파일이 없습니다.")